In [2]:
import os
import pandas as pd
from sklearn.impute import KNNImputer

# Directories
input_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/sorted_Continuous_Data'
output_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Imputed_Data_KNN'

# Ensure the output directory exists
os.makedirs(output_directory, exist_ok=True)

# List all CSV files in the input directory
csv_files = [f for f in os.listdir(input_directory) if f.endswith('.csv')]

# Columns to exclude from imputation (modifiable)
exclude_columns = ['PM2.5']  # Add or remove columns as needed

# Function to apply KNN imputation
def apply_knn_imputation(data, n_neighbors=5):
    imputer = KNNImputer(n_neighbors=n_neighbors)
    imputed_data = pd.DataFrame(imputer.fit_transform(data), columns=data.columns)
    return imputed_data

# Iterate through each CSV file and perform KNN imputation
for csv_file in csv_files:
    input_file_path = os.path.join(input_directory, csv_file)
    data = pd.read_csv(input_file_path)

    # Extract columns to be excluded from imputation
    excluded_data = data[exclude_columns]
    
    # Apply KNN imputation to all columns except those in exclude_columns and 'datetime'
    columns_to_impute = [col for col in data.columns if col not in exclude_columns + ['datetime']]
    data_without_excluded_datetime = data.drop(columns=['datetime'] + exclude_columns)
    
    # Check for columns with all missing values and fill them with zeros (or any other appropriate value) before imputation
    data_without_excluded_datetime.fillna(0, inplace=True)

    # Perform KNN imputation
    imputed_data = apply_knn_imputation(data_without_excluded_datetime)
    
    # Ensure the number of columns match
    if imputed_data.shape[1] != data_without_excluded_datetime.shape[1]:
        print(f"Column mismatch detected in {csv_file}. Imputed columns: {imputed_data.shape[1]}, Original columns: {data_without_excluded_datetime.shape[1]}")
        continue
    
    # Add back excluded columns and 'datetime'
    imputed_data.insert(0, 'datetime', data['datetime'])
    imputed_data = pd.concat([imputed_data, excluded_data], axis=1)

    # Save the resulting DataFrame to the output folder
    output_file_path = os.path.join(output_directory, csv_file)
    imputed_data.to_csv(output_file_path, index=False)

    print(f"Processed and saved: {csv_file}")

print("KNN imputation complete for all files.")


Processed and saved: AQMS_NEWCASTLE_2018-12-31_2024-07-16_data.csv
Processed and saved: AQMS_PARRAMATTA_2018-12-31_2024-07-16_data.csv
Processed and saved: AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv
Processed and saved: AQMS_LIVERPOOL_2018-12-31_2024-07-16_data.csv
Processed and saved: AQMS_ARMIDALE_2018-12-31_2024-07-16_data.csv
Processed and saved: AQMS_WAGGA_2018-12-31_2024-07-16_data.csv
Processed and saved: AQMS_WOLLONGONG_2018-12-31_2024-07-16_data.csv
Processed and saved: AQMS_BATHURST_2018-12-31_2024-07-16_data.csv
KNN imputation complete for all files.
